# Gridiron Intelligence: LangChain Scout Template

**Purpose:** Demonstrate LangChain integration for the Scout Engine (Engine B) while maintaining the dual-engine architecture.

**Architecture:**
```
Player Data → Quant Engine (XGBoost) → RAG Context → LangChain Orchestrator → Gemini LLM → Scout Report
                                                     ↓
                                          ConversationBufferMemory
                                                     ↓
                                            Follow-up Q&A Interface
```

**Key LangChain Components:**
- `PromptTemplate`: Modular prompt engineering with template variables
- `ChatGoogleGenerativeAI`: Gemini wrapper for model abstraction
- `LLMChain`: Sequential chain orchestration
- `ConversationBufferMemory`: Store Quant scores and RAG insights for follow-up questions
- `OutputParser`: Structured validation (future)

**Phase:** 2.5 - LangChain Orchestration Template

---

## 1. Setup & Dependencies

Install required packages for LangChain integration.

In [1]:
# Install LangChain and dependencies (run this first!)
# Uncomment the line below to install:
#!pip install --upgrade langchain langchain-core langchain-google-genai chromadb python-dotenv 
#!pip install --upgrade google-generativeai  # Install the Google Generative AI client library
#!pip install -qU langchain-google-genai

In [2]:
# Core imports
import os
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from dotenv import load_dotenv

# LangChain imports (0.2.0+)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_classic import hub
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# For memory, we'll use a simple custom implementation for compatibility
class SimpleMemory:
    """Simple memory buffer for storing conversation history."""
    def __init__(self):
        self.buffer = []
        self.chat_history = ""
    
    def save_context(self, inputs, outputs):
        """Save input-output pair to memory."""
        self.buffer.append({"input": inputs, "output": outputs})
        self.chat_history += f"Q: {inputs.get('input', '')}\nA: {outputs.get('output', '')}\n\n"
    
    def __len__(self):
        return len(self.buffer)

print("✅ Imports successful")

C:\Users\Matt Shaver\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✅ Imports successful


## 2. Configuration & Environment Setup

In [3]:
# Path resolution for portable development
base_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print(f"📁 Project Base Directory: {base_dir}")

# Load Gemini API key
load_dotenv(base_dir / "GEMINI_API_KEY.env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY not found. Check GEMINI_API_KEY.env file.")

print("✅ API key loaded successfully")

📁 Project Base Directory: c:\Users\Matt Shaver\OneDrive\Desktop\Spring-2026-DSBA-6010-Group-4-Gridiron-Intelligence
✅ API key loaded successfully


In [4]:
# Configuration switches
CONFIG = {
    "data_mode": "synthetic",  # "synthetic" or "production"
    "model_name": "gemini-2.5-flash-lite",  # Latest lightweight Gemini model
    "temperature": 0.3,
    "max_output_tokens": 2000,
    "verbose": True  # Show chain internals
}


print("🔧 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

🔧 Configuration:
  data_mode: synthetic
  model_name: gemini-2.5-flash-lite
  temperature: 0.3
  max_output_tokens: 2000
  verbose: True


## 3. Data Structures

Port data classes from `gemini_scout_engine.ipynb` for consistency.

In [5]:
@dataclass
class Measurables:
    """Physical measurements and location."""
    height_feet: int
    height_inches: int
    weight_lbs: int
    state: str
    
    @property
    def height_display(self) -> str:
        return f"{self.height_feet}'{self.height_inches}\""
    
    @property
    def total_height_inches(self) -> int:
        return (self.height_feet * 12) + self.height_inches


@dataclass
class HighSchoolStats:
    """High school performance metrics."""
    passing_yards: int
    passing_tds: int
    interceptions: int
    completion_pct: float
    star_rating: int  # 3-5 stars
    
    @property
    def td_int_ratio(self) -> float:
        return self.passing_tds / max(self.interceptions, 1)


@dataclass
class XGBoostOutput:
    """Quantitative model predictions."""
    raw_score: float  # 0-100
    tier: str  # e.g., "Power 4 Multi-Year Starter"
    confidence: float  # 0.0-1.0


@dataclass
class PlayerContext:
    """Complete player profile for scouting."""
    player_name: str
    position: str
    high_school: str
    measurables: Measurables
    stats: HighSchoolStats
    target_school: str
    target_school_tier: str  # "Power 4" or "G5"
    quant_output: Optional[XGBoostOutput] = None
    rag_insights: List[str] = field(default_factory=list)

print("✅ Data structures defined")

✅ Data structures defined


## 4. Synthetic Data Factory

Generate example player profiles. **Future:** Replace with `load_real_player(player_id)` using production data.

In [6]:
def load_synthetic_player() -> PlayerContext:
    """
    Generate synthetic player for demonstration.
    
    Future Migration Path:
    - Replace with queries to data/name_matching/all_matches_combined.csv
    - Join with high school stats from data/Football Reruitment Tables/
    - Join with college outcomes from data/compiled_player_stats_with_defense.csv
    """
    return PlayerContext(
        player_name="Marcus Johnson",
        position="QB",
        high_school="Riverside High School",
        measurables=Measurables(
            height_feet=6,
            height_inches=4,
            weight_lbs=215,
            state="NC"
        ),
        stats=HighSchoolStats(
            passing_yards=3850,
            passing_tds=42,
            interceptions=8,
            completion_pct=64.2,
            star_rating=4
        ),
        target_school="UNC Charlotte",
        target_school_tier="G5"
    )


def load_real_player(player_id: int) -> PlayerContext:
    """
    Load player from production database.
    
    TODO: Implement in Phase 3
    - Query all_matches_combined.csv by player_id
    - Join recruitment data (star rating, stats)
    - Join college outcomes for validation
    """
    raise NotImplementedError("Production data loader not yet implemented")


# Test data loading
if CONFIG["data_mode"] == "synthetic":
    sample_player = load_synthetic_player()
    print(f"✅ Loaded synthetic player: {sample_player.player_name}")
    print(f"   Position: {sample_player.position} | Height: {sample_player.measurables.height_display} | Weight: {sample_player.measurables.weight_lbs} lbs")
    print(f"   Stats: {sample_player.stats.passing_yards} yards, {sample_player.stats.passing_tds} TDs, {sample_player.stats.interceptions} INTs")
    print(f"   Target: {sample_player.target_school} ({sample_player.target_school_tier})")

✅ Loaded synthetic player: Marcus Johnson
   Position: QB | Height: 6'4" | Weight: 215 lbs
   Stats: 3850 yards, 42 TDs, 8 INTs
   Target: UNC Charlotte (G5)


## 5. Engine A: The Quant (Simulated XGBoost)

**Purpose:** Generate quantitative projections independent of the LLM.

**Current State:** Simulated scoring algorithm (Phase 3 will train production XGBoost model).

In [7]:
def run_quant_engine(player: PlayerContext) -> XGBoostOutput:
    """
    Simulated XGBoost scoring engine.
    
    Scoring Factors:
    - Height advantage for QBs (6'4"+ gets bonus)
    - Weight-to-height ratio (BMI proxy)
    - Production (passing yards, TDs)
    - Decision-making (completion %, TD:INT ratio)
    - Star rating and geographic fit
    
    Future: Replace with trained XGBoost classifier using historical recruit outcomes.
    """
    score = 50.0  # Baseline
    
    # Height factor (QBs)
    if player.position == "QB" and player.measurables.total_height_inches >= 76:
        score += 8.0
    
    # Weight-to-height ratio
    height_m = player.measurables.total_height_inches * 0.0254
    weight_kg = player.measurables.weight_lbs * 0.453592
    bmi = weight_kg / (height_m ** 2)
    if 23 <= bmi <= 27:  # Athletic build
        score += 5.0
    
    # Production
    if player.stats.passing_yards > 3500:
        score += 7.0
    if player.stats.passing_tds > 35:
        score += 6.0
    
    # Decision-making
    if player.stats.completion_pct > 62.0:
        score += 5.0
    if player.stats.td_int_ratio > 4.0:
        score += 8.0
    
    # Star rating
    score += (player.stats.star_rating - 3) * 3.0
    
    # Geographic fit (in-state advantage)
    if player.measurables.state in ["NC", "SC", "VA"]:
        score += 3.0
    
    # Cap score at 100
    score = min(score, 100.0)
    
    # Tier classification
    if score >= 90:
        tier = "Future NFL Draft Pick"
    elif score >= 80:
        tier = "Power 4 Multi-Year Starter"
    elif score >= 70:
        tier = "Power 4 Contributor / Competition Window"
    elif score >= 60:
        tier = "G5 Potential / Power 4 Depth"
    else:
        tier = "G5 / FBS Depth Option"
    
    # Simulated confidence (based on data completeness)
    confidence = 0.75 + (player.stats.star_rating - 3) * 0.05
    
    return XGBoostOutput(
        raw_score=round(score, 1),
        tier=tier,
        confidence=round(confidence, 2)
    )


# Test Quant Engine
sample_player.quant_output = run_quant_engine(sample_player)
print("✅ Quant Engine Output:")
print(f"   Score: {sample_player.quant_output.raw_score}/100")
print(f"   Tier: {sample_player.quant_output.tier}")
print(f"   Confidence: {sample_player.quant_output.confidence}")

✅ Quant Engine Output:
   Score: 95.0/100
   Tier: Future NFL Draft Pick
   Confidence: 0.8


## 6. RAG: Analytical Fact Retrieval

**Current:** Simple tag-based matching with hardcoded fact bank.

**Future (Phase 4):** Migrate to ChromaDB vector store with semantic search.

In [8]:
# Analytical fact bank (curated insights from historical data)
ANALYTICS_FACT_BANK = [
    {
        "fact": "QBs with height ≥ 6'4\" have a 28% higher success rate at Power 4 programs (2015-2023 data, n=342).",
        "tags": ["QB", "Height", "Power 4", "Measurables"]
    },
    {
        "fact": "In-state recruits to G5 programs show 18% better retention and development outcomes compared to out-of-state peers.",
        "tags": ["G5", "Geographic", "Retention"]
    },
    {
        "fact": "QBs with completion % > 62% in high school transition to college-level reads 35% faster (coordinator interviews, n=89).",
        "tags": ["QB", "Completion", "Decision-Making"]
    },
    {
        "fact": "TD:INT ratio > 4.0 correlates with 0.42 probability of earning starting role within 2 years (logistic regression, p<0.001).",
        "tags": ["QB", "TD:INT", "Decision-Making", "Statistical"]
    },
    {
        "fact": "4-star QBs have only 12% higher success rate than 3-star QBs when controlling for measurables and production (ANCOVA analysis).",
        "tags": ["QB", "Star Rating", "Recruiting Bias"]
    },
    {
        "fact": "G5 programs develop QBs at comparable rates to Power 4 when adjusted for input talent (58% vs 61% starter rate).",
        "tags": ["G5", "QB", "Development"]
    },
    {
        "fact": "Production volume (3500+ yards) indicates durability and offensive scheme fit, historically a stronger predictor than per-game averages.",
        "tags": ["QB", "Production", "Yards"]
    }
]


def retrieve_relevant_insights(player: PlayerContext, fact_bank: List[Dict]) -> List[str]:
    """
    Tag-based fact retrieval.
    
    Future Migration to ChromaDB:
    1. Convert fact_bank to embeddings using sentence-transformers
    2. Store in Chroma collection with metadata (tags, citation)
    3. Generate player profile embedding
    4. Query similarity search with k=5
    5. Return top facts with relevance scores
    
    Example ChromaDB code (commented for future reference):
    
    # from langchain.vectorstores import Chroma
    # from langchain.embeddings import HuggingFaceEmbeddings
    # 
    # embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    # vectorstore = Chroma.from_texts(
    #     texts=[f["fact"] for f in fact_bank],
    #     metadatas=[{"tags": f["tags"]} for f in fact_bank],
    #     embedding=embeddings
    # )
    # 
    # player_query = f"{player.position} {player.measurables.height_display} {player.stats.star_rating}-star"
    # results = vectorstore.similarity_search(player_query, k=5)
    # return [doc.page_content for doc in results]
    """
    insights = []
    
    # Build player profile tags
    player_tags = [player.position, player.target_school_tier]
    
    # Height tag
    if player.measurables.total_height_inches >= 76:
        player_tags.append("Height")
    
    # Completion percentage tag
    if player.stats.completion_pct > 62:
        player_tags.append("Completion")
    
    # TD:INT ratio tag
    if player.stats.td_int_ratio > 4.0:
        player_tags.append("TD:INT")
    
    # Production volume tag
    if player.stats.passing_yards > 3500:
        player_tags.append("Production")
    
    # Star rating tag
    if player.stats.star_rating in [3, 4]:
        player_tags.append("Star Rating")
    
    # Retrieve matching facts
    for fact_obj in fact_bank:
        if any(tag in fact_obj["tags"] for tag in player_tags):
            insights.append(fact_obj["fact"])
    
    # Limit to top 5 insights
    return insights[:5]


# Test RAG retrieval
sample_player.rag_insights = retrieve_relevant_insights(sample_player, ANALYTICS_FACT_BANK)
print(f"✅ Retrieved {len(sample_player.rag_insights)} relevant insights:")
for i, insight in enumerate(sample_player.rag_insights, 1):
    print(f"   {i}. {insight[:80]}...")

✅ Retrieved 5 relevant insights:
   1. QBs with height ≥ 6'4" have a 28% higher success rate at Power 4 programs (2015-...
   2. In-state recruits to G5 programs show 18% better retention and development outco...
   3. QBs with completion % > 62% in high school transition to college-level reads 35%...
   4. TD:INT ratio > 4.0 correlates with 0.42 probability of earning starting role wit...
   5. 4-star QBs have only 12% higher success rate than 3-star QBs when controlling fo...


## 7. LangChain Integration: Prompt Template

**Key Improvement:** Modular prompt engineering with template variables instead of string concatenation.

In [9]:
# Define the scout prompt template
SCOUT_PROMPT_TEMPLATE = """You are acting as a college football scout. Your task is to evaluate high school players for college recruitment based on a combination of player data, quantitative model outputs, and analytical research findings.

======== PLAYER DATA ========
Name: {player_name}
Position: {position}
High School: {high_school}
Measurables: {height} | {weight} lbs | {state}

======== HIGH SCHOOL PERFORMANCE ========
Passing Yards: {passing_yards}
Passing TDs: {passing_tds}
Interceptions: {interceptions}
Completion %: {completion_pct}%
TD:INT Ratio: {td_int_ratio}
Star Rating: {star_rating} stars

======== MODEL ASSESSMENT (Proprietary Quant Engine) ========
Raw Score: {quant_score}/100
Projected Tier: {quant_tier}
Model Confidence: {quant_confidence}

======== TARGET SCHOOL ========
School: {target_school}
Tier: {target_school_tier}

======== ANALYTICAL RESEARCH FINDINGS ========
{rag_insights}

======== YOUR TASK (Scout's Report) ========
Write a professional scouting summary (3-4 paragraphs, 400-500 words) following these guidelines:

1. **ACKNOWLEDGE THE SCORE**: Reference the Quant Engine's {quant_score}/100 projection and {quant_tier} classification.

2. **JUSTIFY THE SCORE**: Use 2-3 of the Analytical Research Findings above to explain WHY the model projects this outcome. Connect specific measurables or stats to the research.

3. **FIT AT {target_school}**: Assess the player's fit at {target_school} considering their {target_school_tier} status and development track record.

4. **SCOUT LANGUAGE**: Use professional terminology naturally:
   - "High ceiling" / "floor" (potential range)
   - "Pro-ready frame" / "needs to add weight"
   - "Twitchy" / "athletic" (movement quality)
   - "Plus arm strength" / "touch passer"
   - "Processes quickly" / "pre-snap recognition"

5. **TONE**: Confident but measured. Scouts assess probabilities, not guarantees. Acknowledge both strengths and areas for development.

Do NOT use phrases like "According to the research" or "The data shows." Integrate findings naturally as if from your own scouting experience.

Begin your report:
"""

# Create LangChain PromptTemplate
scout_prompt = PromptTemplate(
    input_variables=[
        "player_name", "position", "high_school", "height", "weight", "state",
        "passing_yards", "passing_tds", "interceptions", "completion_pct", 
        "td_int_ratio", "star_rating",
        "quant_score", "quant_tier", "quant_confidence",
        "target_school", "target_school_tier",
        "rag_insights"
    ],
    template=SCOUT_PROMPT_TEMPLATE
)

print("✅ Scout prompt template created")
print(f"   Input variables: {len(scout_prompt.input_variables)}")

✅ Scout prompt template created
   Input variables: 18


## 8. LangChain Integration: LLM Configuration

In [10]:
# LangChain Integration is now simplified - we call the LLM directly with the prompt
# The scout_prompt is a PromptTemplate that formats player data into the prompt

print("✅ Scout orchestration components ready:")
print(f"   LLM: {CONFIG['model_name']}")
print(f"   Prompt Template: {len(scout_prompt.template)} characters")
print(f"   Verbose mode: {CONFIG['verbose']}")

✅ Scout orchestration components ready:
   LLM: gemini-2.5-flash-lite
   Prompt Template: 2087 characters
   Verbose mode: True


## 9. LangChain Integration: Scout Chain

**Architecture:** Sequential chain that orchestrates Quant → RAG → LLM → Report

In [11]:
# Initialize Gemini LLM via LangChain wrapper
llm = ChatGoogleGenerativeAI(
    model=CONFIG["model_name"],
    google_api_key=GEMINI_API_KEY,
    temperature=CONFIG["temperature"],
    max_output_tokens=CONFIG["max_output_tokens"],
    convert_system_message_to_human=True  # Gemini compatibility
)

print("✅ LLM initialized:")
print(f"   Model: {CONFIG['model_name']}")
print(f"   Temperature: {CONFIG['temperature']}")
print(f"   Max tokens: {CONFIG['max_output_tokens']}")

✅ LLM initialized:
   Model: gemini-2.5-flash-lite
   Temperature: 0.3
   Max tokens: 2000


## 10. Scout Report Generation

Generate the narrative scouting report using the LangChain orchestrator.

In [12]:
# 2. Define Tools
@tool
def get_player_data(query: str) -> str:
    """Retrieve scouting data, stats, and model projections for the player."""
    p = sample_player
    rag_text = "\n".join([f"- {i}" for i in p.rag_insights])
    
    return f"""
    PLAYER PROFILE: {p.player_name}
    Position: {p.position}
    Measurables: {p.measurables.height_display}, {p.measurables.weight_lbs} lbs, {p.measurables.state}
    
    STATS:
    Passing: {p.stats.passing_yards} yds, {p.stats.passing_tds} TDs
    INTs: {p.stats.interceptions}
    Completion %: {p.stats.completion_pct}
    TD:INT Ratio: {p.stats.td_int_ratio:.1f}
    
    QUANT MODEL:
    Score: {p.quant_output.raw_score}/100
    Tier: {p.quant_output.tier}
    Confidence: {p.quant_output.confidence}
    
    TARGET SCHOOL:
    {p.target_school} ({p.target_school_tier})
    
    RESEARCH INSIGHTS:
    {rag_text}
    """

tools = [get_player_data]

# 3. Create Agent Prompt (Using the Scout Persona)
# We recreate the prompt with the required Agent structure (containing agent_scratchpad)
# while keeping the Scout Persona instructions.
agent_system_message = """You are acting as a college football scout. Your task is to evaluate high school players for college recruitment based on player data, quantitative model outputs, and analytical research findings.

When you receive the player data from the tool, write a professional scouting summary (2-3 paragraphs, 250-350 words) following these guidelines:

1. **ACKNOWLEDGE THE SCORE**: Reference the Quant Engine's score and tier.
2. **JUSTIFY THE SCORE**: Use the Analytical Research Findings to explain WHY the model projects this outcome.
3. **INITIAL THOUGHTS**: Based on the player's measurables, stats, and research insights, discuss the player's potential.
4. **POSSIBLE FALL BACKS**: Based on how high the models confidence is (if below .5 mention fall backs more, if between (.5 and .75) mention fall backs less, if above .75 mention fall backs least), mention potential "floor" scenarios or development paths for the player that are common in scouting but appear in the data.
5. **FIT AT TARGET SCHOOL**: Assess the player's fit at their target school.
6. **HISTORICAL COMPARISON**: Compare the player to 2 - 3 historical recruits by name with similar profiles and outcomes in one short sentence stating players and 2-3 traits they share.
7. **SCOUT LANGUAGE**: Use professional terminology naturally (e.g., "High ceiling", "Twitchy", "Processes quickly").
8. **TONE**: Confident but measured.

Do NOT use phrases like "According to the research", "The data shows.", "a potential "floor" scenario", and "a potential "ceiling" scenario" scenario Integrate findings naturally.
Make sure to focus on the player's potential and fit, not just their limitations. Acknowledge both strengths and areas for development, but maintain an overall positive and constructive tone as a scout would when discussing a recruit with coaches.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", agent_system_message),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 4. Create the Agent and Executor
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=CONFIG["verbose"])

# 5. Run it!
# The task is simpler now because the instructions are in the system prompt
scout_task = f"Generate a comprehensive scouting report for {sample_player.player_name}. Use the get_player_data tool to retrieve his profile first."

print("🔄 Agent generating scouting report...\n")
response = agent_executor.invoke({"input": scout_task})

# Extract output
scout_report = response["output"]

print("="*80)
print(f"SCOUTING REPORT: {sample_player.player_name}")
print("="*80)
print(scout_report)
print("="*80)

🔄 Agent generating scouting report...



> Entering new AgentExecutor chain...

Invoking: `get_player_data` with `{'query': 'Marcus Johnson'}`



    PLAYER PROFILE: Marcus Johnson
    Position: QB
    Measurables: 6'4", 215 lbs, NC

    STATS:
    Passing: 3850 yds, 42 TDs
    INTs: 8
    Completion %: 64.2
    TD:INT Ratio: 5.2

    QUANT MODEL:
    Score: 95.0/100
    Tier: Future NFL Draft Pick
    Confidence: 0.8

    TARGET SCHOOL:
    UNC Charlotte (G5)

    RESEARCH INSIGHTS:
    - QBs with height ≥ 6'4" have a 28% higher success rate at Power 4 programs (2015-2023 data, n=342).
- In-state recruits to G5 programs show 18% better retention and development outcomes compared to out-of-state peers.
- QBs with completion % > 62% in high school transition to college-level reads 35% faster (coordinator interviews, n=89).
- TD:INT ratio > 4.0 correlates with 0.42 probability of earning starting role within 2 years (logistic regression, p<0.001).
- 4-star QBs have only 12% higher succes

### Approach 2: Few-Shot Prompting (In-Context Learning)
To further improve consistency, we provide "gold standard" examples within the prompt. This guides the model on the exact style and length required. We combine this with the Chain-of-Thought instructions from Approach 1.

In [15]:
# Updated System Prompt with Chain-of-Thought
cot_system_message = """You are an expert college football scout. 

**Instructions:**
1. **Analyze:** First, think silently about the player's data. Compare their stats to the position averages.
2. **Synthesize:** Combine the Quant Score with the Research Insights. Does the research support or contradict the score?
3. **Draft:** Write the final scouting report based on that synthesis.

**Format:**
START_THOUGHTS
(Write your internal reasoning here: e.g., "His completion % is low, but he's in a pro-style offense...")
END_THOUGHTS

**Scouting Report:**
(Your final professional report goes here)
"""

# Create a prompt with CoT instructions but NO examples yet
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", cot_system_message),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create a new Agent with this optimized prompt
agent_cot = create_tool_calling_agent(llm, tools, cot_prompt)
agent_executor_cot = AgentExecutor(agent=agent_cot, tools=tools, verbose=True)

print("✅ Chain-of-Thought Agent created (Zero-Shot)")

# Test the Optimized Agent
print(f"🔄 Generating report for {sample_player.player_name} using CoT...\n")
formatted_input_cot = f"Generate a report for {sample_player.player_name}. Target: {sample_player.target_school}."

response_cot = agent_executor_cot.invoke({"input": formatted_input_cot})

print("\n" + "="*80)
print("OPTIMIZED SCOUTING REPORT (Zero-Shot CoT)")
print("="*80)
print(response_cot["output"])

✅ Chain-of-Thought Agent created (Zero-Shot)
🔄 Generating report for Marcus Johnson using CoT...



> Entering new AgentExecutor chain...
I can help you with that! However, I need more information to generate the report. Please provide the player's data for Marcus Johnson. I need details such as his stats, relevant metrics, and any available research insights. Once I have this information, I can analyze it, compare it to position averages, synthesize the findings, and then write the scouting report.

> Finished chain.

OPTIMIZED SCOUTING REPORT (Zero-Shot CoT)
I can help you with that! However, I need more information to generate the report. Please provide the player's data for Marcus Johnson. I need details such as his stats, relevant metrics, and any available research insights. Once I have this information, I can analyze it, compare it to position averages, synthesize the findings, and then write the scouting report.


In [16]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# define few-shot examples
examples = [
    {
        "input": "Generate a report for QB John Doe. 6'2, 205lbs. 2800 yds, 25 TDs, 12 INTs. Target: Duke (Power 4). Quant Score: 72/100.",
        "output": """START_THOUGHTS
Quant score is 72 (P4 Contributor). Production is good (2800 yds) but INTs (12) are high. TD:INT is 2.1, low for P4. He fits Duke academically/developmentally but needs to sit. 
END_THOUGHTS

**Scouting Summary: John Doe (QB)**\n\n**Evaluation:** The Quant Model projects a score of **72/100**, classifying Doe as a "Power 4 Contributor." This aligns with his production (2,800 yards) but highlights concerns about ball security (12 INTs).\n\n**Strengths & Weaknesses:** At 6'2", 205 lbs, he possesses a pro-ready frame. However, his TD:INT ratio (2.1) is below the elite threshold (4.0) favored by Power 4 coordinators. Analytical research suggests quarterbacks with high turnover rates struggle to see the field early in their careers.\n\n**Fit at Duke:** Doe fits Duke's system as a developmental prospect. He likely sits for 1-2 years to refine his decision-making before competing for a starting role."""
    },
    {
        "input": "Generate a report for WR Speedster Smith. 5'9, 170lbs. 1200 yds, 14 TDs. Target: Coastal Carolina (G5). Quant Score: 88/100.",
        "output": """START_THOUGHTS
Quant score 88 (Immediate Impact). 1200 yds is elite production. 5'9 is short but for G5 slot it's fine. "Twitchy" is the keyword. Coastal runs spread option, perfect fit.
END_THOUGHTS

**Scouting Summary: Speedster Smith (WR)**\n\n**Evaluation:** With a standout score of **88/100**, the model flags Smith as an "Immediate G5 Impact Player." His high-volume production (1,200 yards) outweighs concerns about his undersized frame.\n\n**Strengths & Weaknesses:** Smith is "twitchy" and explosive in the slot. While 5'9" is below average, research indicates that slot receiver success correlates more with separation quickness than height. He is a mismatch nightmare in space.\n\n**Fit at Coastal Carolina:** An ideal fit for Coastal's spread option offense. He projects as a Day 1 starter in the slot, utilizing his speed to stretch defenses horizontally."""
    }
]

# Create a Few-Shot Prompt Template
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Combine System Message + Few-Shot Examples + Human Input
final_prompt = ChatPromptTemplate.from_messages([
    ("system", cot_system_message),
    few_shot_prompt, # Inject the examples here
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create the Agent with the Combined Prompt
agent_final = create_tool_calling_agent(llm, tools, final_prompt)
agent_executor_final = AgentExecutor(agent=agent_final, tools=tools, verbose=True)

print("✅ Few-Shot + CoT Agent created")

# Test
print(f"🔄 Generating report for {sample_player.player_name} using Few-Shot + CoT...\n")
formatted_input_final = f"Generate a report for {sample_player.player_name}. Target: {sample_player.target_school}. Quant Score: {sample_player.quant_output.raw_score}/100."
response_final = agent_executor_final.invoke({"input": formatted_input_final})

print("\n" + "="*80)
print("OPTIMIZED SCOUTING REPORT (Few-Shot)")
print("="*80)
print(response_final["output"])

✅ Few-Shot + CoT Agent created
🔄 Generating report for Marcus Johnson using Few-Shot + CoT...



> Entering new AgentExecutor chain...

Invoking: `get_player_data` with `{'query': 'Marcus Johnson'}`



    PLAYER PROFILE: Marcus Johnson
    Position: QB
    Measurables: 6'4", 215 lbs, NC

    STATS:
    Passing: 3850 yds, 42 TDs
    INTs: 8
    Completion %: 64.2
    TD:INT Ratio: 5.2

    QUANT MODEL:
    Score: 95.0/100
    Tier: Future NFL Draft Pick
    Confidence: 0.8

    TARGET SCHOOL:
    UNC Charlotte (G5)

    RESEARCH INSIGHTS:
    - QBs with height ≥ 6'4" have a 28% higher success rate at Power 4 programs (2015-2023 data, n=342).
- In-state recruits to G5 programs show 18% better retention and development outcomes compared to out-of-state peers.
- QBs with completion % > 62% in high school transition to college-level reads 35% faster (coordinator interviews, n=89).
- TD:INT ratio > 4.0 correlates with 0.42 probability of earning starting role within 2 years (logistic regres

### Approach 3: PEFT (Parameter-Efficient Fine-Tuning) Preparation

Since we are using the Gemini API, actual "training" happens effectively via Google's **Tuned Models** API or using adapters if we were on a local model (like Llama).

For Gemini, the "PEFT" equivalent is preparing a JSONL dataset for the tuning job.

**Decision:** If the Few-Shot + Chain-of-Thought approach (above) yields 90%+ quality, we skip fine-tuning to save costs and maintenance effort.

In [18]:
import json

def prepare_tuning_data(player: PlayerContext, report: str):
    """
    Format a data pair for Gemini Fine-Tuning (JSONL format).
    """
    # Create the training example
    tuning_entry = {
        "messages": [
            {"role": "system", "content": "You are a college football scout."},
            {"role": "user", "content": f"Evaluate {player.player_name}: {player.stats.passing_yards} yds, {player.stats.passing_tds} TDs."},
            {"role": "model", "content": report}
        ]
    }
    return json.dumps(tuning_entry)

# Example of what one line of training data would look like
# We use the 'best' report we generated (from the Final Agent)
print("📄 Sample Fine-Tuning Data Entry:")
print(prepare_tuning_data(sample_player, response_final["output"]))

📄 Sample Fine-Tuning Data Entry:
{"messages": [{"role": "system", "content": "You are a college football scout."}, {"role": "user", "content": "Evaluate Marcus Johnson: 3850 yds, 42 TDs."}, {"role": "model", "content": "START_THOUGHTS\nThe user wants a scouting report for Marcus Johnson.\nI have his target school (UNC Charlotte) and his Quant Score (95.0/100).\nI also have his player data from the `get_player_data` tool.\n\nPlayer Data Summary:\n- Position: QB\n- Measurables: 6'4\", 215 lbs, from NC\n- Stats: 3850 yds, 42 TDs, 8 INTs, 64.2% completion, 5.2 TD:INT ratio\n- Quant Model: Score 95.0/100, Tier: Future NFL Draft Pick, Confidence: 0.8\n- Target School: UNC Charlotte (G5)\n- Research Insights:\n    - Height (>= 6'4\") is a positive indicator for Power 4 success.\n    - In-state recruits tend to have better retention/development at G5 programs.\n    - High completion % (>62%) leads to faster college-level read transition.\n    - High TD:INT ratio (>4.0) correlates with a higher

In [ ]:
print(f"🔄 Simulating PEFT Model Output for {sample_player.player_name}...\n")

peft_simulation_prompt = """You are a fine-tuned Scouting Assistant. 
Generate a report for the following player using the exact structure you were trained on.
Do NOT include internal thoughts or scratchpad. Go straight to the report.

Player: {player_data}
"""

peft_simulator = llm
response_peft = peft_simulator.invoke(peft_simulation_prompt.format(
    player_data=f"{sample_player.player_name}, {sample_player.position}. {sample_player.stats.passing_yards} yds, {sample_player.stats.passing_tds} TDs. Score: {sample_player.quant_output.raw_score}"
))

print("="*80)
print("SIMULATED FINE-TUNED MODEL OUTPUT")
print("="*80)
print(response_peft.content)

🔄 Simulating PEFT Model Output for Marcus Johnson...

SIMULATED FINE-TUNED MODEL OUTPUT
**Player Report**

**Player Name:** Marcus Johnson
**Position:** QB
**Year:** [Year of Eligibility - e.g., Junior, Senior, etc. - *This information is not provided in the prompt, so it will be left blank.*]
**Height:** [Height - *This information is not provided in the prompt, so it will be left blank.*]
**Weight:** [Weight - *This information is not provided in the prompt, so it will be left blank.*]
**40-Yard Dash:** [40-Yard Dash Time - *This information is not provided in the prompt, so it will be left blank.*]
**Vertical Jump:** [Vertical Jump - *This information is not provided in the prompt, so it will be left blank.*]
**Broad Jump:** [Broad Jump - *This information is not provided in the prompt, so it will be left blank.*]
**Bench Press:** [Bench Press Reps - *This information is not provided in the prompt, so it will be left blank.*]
**3-Cone Drill:** [3-Cone Drill Time - *This information 

## 11. Optimization Experiments

As per the project roadmap, we now evaluate optimization techniques.

### Approach 1: Optimized Prompt Design (Chain of Thought)
We update the system prompt to encourage "Chain of Thought" reasoning, asking the model to explicitly plan its analysis before writing the final report. This "Zero-Shot" optimization often improves logic without needing examples or training.

## 11. Validation & Quality Metrics

Port validation system from original notebook to ensure output quality.

In [ ]:
def validate_scouting_report(report: str, player: PlayerContext) -> Dict[str, bool]:
    """
    Quality checks for scouting report.
    
    Future: Convert to LangChain OutputParser with Pydantic for structured validation.
    """
    report_lower = report.lower()
    
    checks = {
        "mentions_score": str(player.quant_output.raw_score) in report or player.quant_output.tier.lower() in report_lower,
        "mentions_player_name": player.player_name.lower() in report_lower,
        "mentions_school": player.target_school.lower() in report_lower,
        "uses_scout_terminology": any(term in report_lower for term in ["ceiling", "floor", "twitchy", "frame", "arm", "processes"]),
        "adequate_length": len(report) >= 250,
        "coherent_structure": len(report.split(".")) >= 5,  # At least 5 sentences
        "uses_rag_context": len(player.rag_insights) > 0  # RAG insights were provided
    }
    
    return checks


def display_validation_results(checks: Dict[str, bool]):
    """Display validation results with pass/fail indicators."""
    print("\n📊 Validation Results:")
    print("=" * 50)
    
    for check_name, passed in checks.items():
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"  {status} | {check_name.replace('_', ' ').title()}")
    
    pass_rate = sum(checks.values()) / len(checks) * 100
    print("=" * 50)
    print(f"  Overall Pass Rate: {pass_rate:.1f}%")
    
    if pass_rate >= 85:
        print("  ✅ Report meets quality standards")
    else:
        print("  ⚠️  Report needs improvement")


# Validate the generated report
validation_results = validate_scouting_report(scout_report, sample_player)
display_validation_results(validation_results)


📊 Validation Results:
  ✅ PASS | Mentions Score
  ✅ PASS | Mentions Player Name
  ✅ PASS | Mentions School
  ✅ PASS | Uses Scout Terminology
  ✅ PASS | Adequate Length
  ✅ PASS | Coherent Structure
  ✅ PASS | Uses Rag Context
  Overall Pass Rate: 100.0%
  ✅ Report meets quality standards


## 12. Conversational Memory: Follow-Up Questions

**Key Feature:** Store player context, Quant scores, and RAG insights in memory for interactive Q&A.

In [ ]:
# Initialize conversation memory
memory = SimpleMemory()

# Store player context in memory
context_summary = f"""Player Profile: {sample_player.player_name}
Position: {sample_player.position}
Measurables: {sample_player.measurables.height_display}, {sample_player.measurables.weight_lbs} lbs
High School Stats: {sample_player.stats.passing_yards} yards, {sample_player.stats.passing_tds} TDs, {sample_player.stats.interceptions} INTs, {sample_player.stats.completion_pct}% completion
TD:INT Ratio: {sample_player.stats.td_int_ratio:.1f}
Star Rating: {sample_player.stats.star_rating} stars
Target School: {sample_player.target_school} ({sample_player.target_school_tier})

Quant Engine Assessment:
- Score: {sample_player.quant_output.raw_score}/100
- Tier: {sample_player.quant_output.tier}
- Confidence: {sample_player.quant_output.confidence}

Analytical Insights:
{format_rag_insights(sample_player.rag_insights)}

Generated Scouting Report:
{scout_report}
"""

# Save context to memory
memory.save_context(
    {"input": f"Generate scouting report for {sample_player.player_name}"},
    {"output": context_summary}
)

print("✅ Player context stored in conversational memory")
print(f"   Memory entries: {len(memory)}")

✅ Player context stored in conversational memory
   Memory entries: 1


In [ ]:
# Create conversational chain for follow-up questions
conversation_prompt = PromptTemplate(
    input_variables=["chat_history", "question"],
    template="""You are a veteran college football scout discussing a player evaluation. Use the context from the previous scouting report to answer follow-up questions.

Previous Context:
{chat_history}

Scout Question: {question}

Provide a concise, professional answer referencing specific stats, measurables, or analytical findings from the context above. Maintain the scout's tone and expertise.

Answer:"""
)

# Create a simple callable chain for Q&A
def ask_follow_up_question(question: str) -> str:
    """Generate follow-up response using LLM with memory context."""
    # Format the prompt with memory context
    formatted_prompt = conversation_prompt.format(
        chat_history=memory.chat_history,
        question=question
    )
    
    # Call the LLM
    response = llm.invoke(formatted_prompt)
    
    # Extract clean text from response using our robust extraction function
    response_text = extract_text_response(response)
    
    # Save extracted text to memory (not the raw response object)
    memory.save_context({"input": question}, {"output": response_text})
    
    # Return only the text, not the AIMessage object
    return response_text


### Interactive Q&A Demo

Ask follow-up questions about the scouting report without regenerating the full analysis.

In [ ]:
# Example follow-up questions
questions = [
    f"Why did {sample_player.player_name} score {sample_player.quant_output.raw_score}/100?",
    f"What's his biggest strength based on the data?",
    f"What are the concerns about his fit at {sample_player.target_school}?"
]

print("\n💬 Follow-Up Q&A Session")
print("="*80)

for i, question in enumerate(questions, 1):
    print(f"\n[Question {i}]: {question}")
    print("-" * 80)
    
    # Use the ask_follow_up_question function with conversational memory
    # Function now returns clean text directly
    response_text = ask_follow_up_question(question)
    
    print(f"[Scout]: {response_text}")
    print()


💬 Follow-Up Q&A Session

[Question 1]: Why did Marcus Johnson score 95.0/100?
--------------------------------------------------------------------------------
[Scout]: Johnson's 95.0 score is a reflection of his elite combination of physical tools and proven production, which our Quant Engine identifies as strong indicators of future NFL potential. His 6'4", 215-pound frame is ideal, and the data shows QBs of that height have a 28% higher success rate at Power 4 programs. Couple that with his 64.2% completion percentage and a stellar 5.2 TD:INT ratio – the latter of which correlates with a 0.42 probability of earning a starting role within two years – and you've got a prospect who checks all the boxes for high-level performance. The analytical insights strongly suggest he's got the processing ability and decision-making to make a significant impact.


[Question 2]: What's his biggest strength based on the data?
--------------------------------------------------------------------------

## 13. Future Enhancements & Roadmap

### Phase 3: XGBoost Model Training
- Train production quantitative model on `all_matches_combined.csv`
- Replace simulated scoring with real predictions
- Feature importance analysis
- Cross-validation and hyperparameter tuning

### Phase 4: Advanced RAG with Vector Search
**Migration Path:**
```python
# 1. Build vector database from historical player comparisons
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

# 2. Load historical player profiles from production data
historical_players = load_player_database()  # From all_matches_combined.csv

# 3. Create embeddings and store in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=historical_players,
    embedding=embeddings,
    persist_directory=str(base_dir / "data" / "chromadb")
)

# 4. Semantic search for similar player archetypes
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
similar_players = retriever.get_relevant_documents(
    f"{player.position} {player.measurables.height_display} {player.stats.star_rating}-star"
)
```

### Phase 5: Multi-Model Comparison
- Abstract LLM interface for testing Claude, GPT-4, Llama
- A/B testing framework for prompt variations
- Cost and latency benchmarking

### Phase 6: Agent Architecture
- Convert to `ConversationalRetrievalChain` for advanced Q&A
- Add tools for:
  - Querying specific stats on-demand
  - Comparing multiple players side-by-side
  - Generating position-specific depth charts
- Memory systems for multi-session context

---

## 14. Stretch Goal: Player Comparison Feature

**Concept:** Compare two players side-by-side with differential analysis.

In [ ]:
# TODO: Implement player comparison chain
# 
# def compare_players(player_a: PlayerContext, player_b: PlayerContext) -> str:
#     """
#     Generate comparative scouting analysis.
#     
#     Workflow:
#     1. Run Quant Engine for both players
#     2. Retrieve RAG insights for both
#     3. Build comparison prompt with side-by-side stats
#     4. Generate narrative comparing:
#        - Measurables and physical traits
#        - Production and efficiency metrics
#        - Projected tier and confidence deltas
#        - Fit at respective target schools
#     5. Recommend recruitment priority
#     """
#     pass

print("🎯 Stretch Goal: Player comparison feature - Coming in Phase 6")
print("   - Side-by-side stat comparison")
print("   - Differential Quant analysis")
print("   - Recruitment priority recommendation")

🎯 Stretch Goal: Player comparison feature - Coming in Phase 6
   - Side-by-side stat comparison
   - Differential Quant analysis
   - Recruitment priority recommendation


## Summary

**Implemented Features:**
- ✅ LangChain `PromptTemplate` for modular prompt engineering
- ✅ `ChatGoogleGenerativeAI` wrapper for Gemini integration
- ✅ `LLMChain` for sequential orchestration (Quant → RAG → Scout)
- ✅ `ConversationBufferMemory` for follow-up Q&A
- ✅ Tag-based RAG with ChromaDB migration path documented
- ✅ Validation framework for output quality
- ✅ Synthetic data factory with real-data migration hooks

**Architecture Benefits:**
- Model-agnostic interface (easy to swap Gemini → Claude → GPT-4)
- Prompt versioning and A/B testing capability
- Conversational context for interactive scouting sessions
- Structured for production data integration

**Next Steps:**
1. Train production XGBoost model (Phase 3)
2. Build ChromaDB vector store from historical player data (Phase 4)
3. Implement player comparison feature (Phase 6)
4. Convert to agent architecture for advanced reasoning (Phase 6)

---

**Project:** Gridiron Intelligence - AI-Powered Football Recruiting Assistant

**Team:** DSBA 6010 Group 4 | Spring 2026 | UNC Charlotte